# Jigsaw | Action relationships, not more topic words

Round 2 is a pre-specified follow-up to an exploratory positive behavior signal. It reuses the four cached round-1 controls. It is not independent confirmation, a new Qwen model, or a Kaggle score. The full round-1 candidate remains rejected.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
while not (ROOT / "configs/relational_features.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "configs/relational_features.json").is_file()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from scripts.run_relational_features import bounded_compute, figures, write_dashboard, verify_prior
config = json.loads((ROOT / "configs/relational_features.json").read_text())
print("Primary comparison:", config["primary"])
print("New fits:", config["new_fits"], " | Reused controls:", config["cached_control_fits"])

Primary comparison: add_act_roles vs add_behavior
New fits: 18  | Reused controls: 4


## 1. Verified starting point

Behavior-only features improved mean policy AUC by 0.02235 in round 1, but the simultaneous interval crossed zero. More scope/rule/support features combined hurt the primary comparison. We do not relabel that failure as success. We retain behavior as an exploratory diagnostic anchor, not a production model.

In [2]:
prior, prior_path = verify_prior(ROOT, config)
display(pd.DataFrame(prior["metrics"])[lambda d: d.variant.isin(["lexical_control", "add_behavior", "full"])])
print("Prior decision:", prior["decision"])
print("Prior outputs remain unchanged:", prior_path)

,fold,policy,variant,auc,brier,log_loss,queries
0,0,"No Advertising: Spam, referral links, unsolici...",lexical_control,0.673022,0.233599,0.667527,234
1,0,"No Advertising: Spam, referral links, unsolici...",add_behavior,0.675784,0.231577,0.663719,234
5,0,"No Advertising: Spam, referral links, unsolici...",full,0.629440,0.263447,0.786720,234
10,1,No legal advice: Do not offer or request legal...,lexical_control,0.641993,0.232466,0.656063,647
11,1,No legal advice: Do not offer or request legal...,add_behavior,0.683940,0.231997,0.710278,647
15,1,No legal advice: Do not offer or request legal...,full,0.664468,0.249920,0.828842,647


Prior decision: DO_NOT_PROMOTE_THIS_FEATURE_SET
Prior outputs remain unchanged: /home/sagemaker-user/projects/jigsaw-rule-classifier/runs/behavioral_features/32e706fe1e7dddb7de34


## 2. Specific new hypotheses

72 observable-text measurements in three 24-column families: actor/action/legal-topic binding; local URL/commercial-intent relationships; and negation/quotation attached to specific actions. Counts and binary indicators are candidates; screening uses eligible training rows only. We also remove legal-topic cues and length cues separately to test whether the earlier behavior gain came from proxies. No external websites or labels are requested.

In [3]:
from scripts.relational_features import CATALOG
display(pd.DataFrame([{"Family": k, "Candidates": len(v)} for k, v in CATALOG.items()]))
result = bounded_compute(ROOT)
print("Decision:", result["decision"])
print("New candidate fits:", result["new_fits"])
print("Reused round-1 controls:", result["reused_round1_controls"])

,Family,Candidates
0,act_roles,24
1,link_intent,24
2,action_scope,24


{"timestamp": "2026-09-11T22:41:09+00:00", "stage": "relational_round2", "event": "started", "elapsed_seconds": 0.0, "stage_elapsed_seconds": 0.0, "total_elapsed_seconds": 0.0}


{"timestamp": "2026-09-11T22:41:15+00:00", "stage": "relational_round2", "event": "candidate_complete", "elapsed_seconds": 5.966, "stage_elapsed_seconds": 5.966, "total_elapsed_seconds": 5.966, "completed": 1, "total": 18, "fold": 0, "variant": "add_act_roles", "new_fits": 1, "reused_fits": 0}


{"timestamp": "2026-09-11T22:41:15+00:00", "stage": "relational_round2", "event": "candidate_complete", "elapsed_seconds": 6.368, "stage_elapsed_seconds": 6.368, "total_elapsed_seconds": 6.368, "completed": 2, "total": 18, "fold": 0, "variant": "add_link_intent", "new_fits": 2, "reused_fits": 0}


{"timestamp": "2026-09-11T22:41:15+00:00", "stage": "relational_round2", "event": "candidate_complete", "elapsed_seconds": 6.775, "stage_elapsed_seconds": 6.775, "total_elapsed_seconds": 6.775, "completed": 3, "total": 18, "fold": 0, "variant": "add_action_scope", "new_fits": 3, "reused_fits": 0}


...

{"timestamp": "2026-09-11T22:41:21+00:00", "stage": "relational_round2", "event": "candidate_complete", "elapsed_seconds": 12.895, "stage_elapsed_seconds": 12.895, "total_elapsed_seconds": 12.895, "completed": 16, "total": 18, "fold": 1, "variant": "without_action_scope", "new_fits": 16, "reused_fits": 0}


{"timestamp": "2026-09-11T22:41:22+00:00", "stage": "relational_round2", "event": "candidate_complete", "elapsed_seconds": 13.136, "stage_elapsed_seconds": 13.136, "total_elapsed_seconds": 13.136, "completed": 17, "total": 18, "fold": 1, "variant": "behavior_without_topic", "new_fits": 17, "reused_fits": 0}


{"timestamp": "2026-09-11T22:41:22+00:00", "stage": "relational_round2", "event": "candidate_complete", "elapsed_seconds": 13.366, "stage_elapsed_seconds": 13.366, "total_elapsed_seconds": 13.366, "completed": 18, "total": 18, "fold": 1, "variant": "behavior_without_length", "new_fits": 18, "reused_fits": 0}


{"timestamp": "2026-09-11T22:41:22+00:00", "stage": "relational_round2", "event": "results_saved", "elapsed_seconds": 13.762, "stage_elapsed_seconds": 13.762, "total_elapsed_seconds": 13.762, "decision": "DO_NOT_PROMOTE_PRIMARY", "new_fits": 18}
{"timestamp": "2026-09-11T22:41:22+00:00", "stage": "relational_round2", "event": "completed", "elapsed_seconds": 13.762, "stage_elapsed_seconds": 13.762, "total_elapsed_seconds": 13.762, "error_type": null}
RESULT: RELATIONAL_ROUND_COMPLETE DECISION: DO_NOT_PROMOTE_PRIMARY


Decision: DO_NOT_PROMOTE_PRIMARY
New candidate fits: 18
Reused round-1 controls: 4


## 3. Per-policy performance and conditional uncertainty

The model, C, lexical vocabulary limits, training weights, and historical 881-query protocol are held fixed. Cached control design parity must pass before any new fit. The primary is **add_act_roles versus add_behavior**; other apparent winners do not substitute for that primary.

In [4]:
display(pd.DataFrame(result["metrics"]))
CHARTS = figures(result)
CHARTS[0].show(renderer="plotly_mimetype")
CHARTS[1].show(renderer="plotly_mimetype")

,fold,policy,variant,auc,brier,log_loss,queries
0,0,"No Advertising: Spam, referral links, unsolici...",lexical_control,0.673022,0.233599,0.667527,234
1,0,"No Advertising: Spam, referral links, unsolici...",add_behavior,0.675784,0.231577,0.663719,234
2,0,"No Advertising: Spam, referral links, unsolici...",add_act_roles,0.675784,0.231472,0.663380,234
3,0,"No Advertising: Spam, referral links, unsolici...",add_link_intent,0.670112,0.235779,0.689949,234
4,0,"No Advertising: Spam, referral links, unsolici...",add_action_scope,0.672276,0.233603,0.668140,234
5,0,"No Advertising: Spam, referral links, unsolici...",full_relations,0.654440,0.242230,0.708172,234
6,0,"No Advertising: Spam, referral links, unsolici...",without_act_roles,0.654515,0.242954,0.710212,234
7,0,"No Advertising: Spam, referral links, unsolici...",without_link_intent,0.671903,0.233076,0.666812,234
8,0,"No Advertising: Spam, referral links, unsolici...",without_action_scope,0.668769,0.235518,0.689242,234
9,0,"No Advertising: Spam, referral links, unsolici...",behavior_without_topic,0.675485,0.231243,0.661525,234


## 4. Matched removal evidence and training-only selection

A full-minus-omitted-family contrast measures conditional contribution in the combined model. Name stability alone does not demonstrate predictive usefulness. No feature is screened on query outcomes.

In [5]:
display(pd.DataFrame(result["comparisons"]))
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

,comparison,delta_auc,simultaneous_low,simultaneous_high,valid_draws
0,add_act_roles vs behavior,0.006238,-0.014359,0.026835,500
1,add_link_intent vs behavior,-0.002650,-0.023247,0.017947,500
2,add_action_scope vs behavior,-0.001719,-0.022317,0.018878,500
3,full_relations vs behavior,-0.004512,-0.025109,0.016085,500
4,without_act_roles vs behavior,-0.010370,-0.030967,0.010227,500
5,without_link_intent vs behavior,0.003940,-0.016657,0.024537,500
6,without_action_scope vs behavior,0.002906,-0.017691,0.023503,500
7,behavior_without_topic vs behavior,-0.022766,-0.043363,-0.002169,500
8,behavior_without_length vs behavior,-0.006538,-0.027135,0.014059,500
9,removal: act_roles,0.005858,-0.014739,0.026455,500


## 5. Mechanism versus proxy

Removing legal-topic and length cues separately from the 58-column behavior bank helps test what drove its observed gain. Each removal re-screens the remaining training columns at the original budget; it is a representation ablation, not coefficient subtraction. Relationships are heuristic and can misparse language. A negated legal directive may still be legal advice; feature firing never assigns moderation labels.

In [6]:
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

## 6. Metric definitions and probability diagnostics

Policy-macro and per-policy-ranked pooled ROC AUC are reported separately. Both are local exploratory metrics; neither is claimed to equal a hidden Kaggle result. Brier and log loss can expose overconfidence even when AUC rises.

In [7]:
display(pd.DataFrame(result["pooled_metrics"]))
CHARTS[6].show(renderer="plotly_mimetype")
CHARTS[7].show(renderer="plotly_mimetype")

,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,lexical_control,0.657508,0.641345,0.650247
1,add_behavior,0.679862,0.680777,0.681784
2,add_act_roles,0.686099,0.691055,0.690944
3,add_link_intent,0.677212,0.679685,0.680523
4,add_action_scope,0.678142,0.679653,0.680908
5,full_relations,0.675349,0.685766,0.685133
6,without_act_roles,0.669491,0.676062,0.676515
7,without_link_intent,0.683802,0.689246,0.689373
8,without_action_scope,0.682768,0.689494,0.689336
9,behavior_without_topic,0.657095,0.646360,0.648485


## 7. Stop/continue decision

The primary must add at least +0.003 mean AUC over the behavior anchor, have a positive simultaneous 95% lower bound, avoid per-policy regressions and not worsen ranked-pooled AUC. The simultaneous band covers the 13 planned contrasts in this round, not adaptive search across all past rounds. Passing only permits consideration of further validation. No automatic GPU run, model promotion, or submission follows.

Research basis: [CheckList](https://aclanthology.org/2020.acl-main.442/) motivates behavioral tests; [NormVio](https://aclanthology.org/2021.findings-emnlp.288/) motivates rule-specific context. [Rule By Example](https://aclanthology.org/2023.acl-long.22/) is contrastive encoder research, **not** what this regex-based CPU experiment implements. The [historical winning solution](https://www.kaggle.com/competitions/jigsaw-agile-community-rules/writeups/1st-place-solution) used support adaptation and ensembles; this feature probe alone does not reproduce it.

In [8]:
display(pd.DataFrame([result["primary_comparison"]]))
print("Primary per-policy deltas:", result["primary_policy_deltas"])
print("Decision:", result["decision"])
for text in result["limitations"]:
    print(text)
print("Interactive dashboard:", write_dashboard(ROOT, result))
print("Private checkpoint directory:", ROOT / "runs/relational_features" / result["run_id"])

,comparison,delta_auc,simultaneous_low,simultaneous_high,valid_draws
0,add_act_roles vs behavior,0.006238,-0.014359,0.026835,500


Primary per-policy deltas: [0.0, 0.01247529402555736]
Decision: DO_NOT_PROMOTE_PRIMARY
Exploratory follow-up chosen after inspecting round 1; not a fresh holdout.
Intervals cover 13 current contrasts, not all adaptive project comparisons.
Heuristic clause/role/negation extraction can be wrong; it never assigns labels.
The CPU behavior anchor is not the accepted Qwen model or a Kaggle score.
No result authorizes GPU work or automatic selection of a different winner.
Interactive dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/relational_features/dashboard.html
Private checkpoint directory: /home/sagemaker-user/projects/jigsaw-rule-classifier/runs/relational_features/b951f663d66c68cb6bca
